# Partie 1 — Exploration des données

Notebook réorganisé selon le TP. Exécuter toutes les cellules de haut en bas.


In [1]:
import pandas as pd
from pathlib import Path

# Détecter le dossier racine, quel que soit le dossier courant de Jupyter
BASE_DIR = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pyproject.toml").exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Ouvrez le projet Retail BI dans VS Code.")


In [2]:
file_path = BASE_DIR / "data/raw/Online Retail.xlsx"

print("Fichier trouvé :", file_path.exists())
print("Chemin utilisé :", file_path.resolve())
if not file_path.exists():
    raise FileNotFoundError("Placez Online Retail.xlsx dans data/raw/.")


Fichier trouvé : True
Chemin utilisé : C:\Users\user\Downloads\Retail_BI_Arborescence_Prof (1)\tp2\data\raw\Online Retail.xlsx


In [3]:
# Lire le fichier Excel
df = pd.read_excel(file_path)
print("Données chargées avec succès !")

Données chargées avec succès !


Comprendre la structure du dataset
*************************************

In [4]:
#   Afficher les premières lignes
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
#   le nombre de lignes et de colonnes
print("Nombre de lignes :", df.shape[0])
print("Nombre de colonnes :", df.shape[1])

Nombre de lignes : 541909
Nombre de colonnes : 8


In [6]:
#   Afficher les noms des colonnes
print("Noms des colonnes :", df.columns.tolist())

Noms des colonnes : ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [7]:
#   Vérifier les types de données
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [8]:
#   Afficher les statistiques descriptives
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,541909.0,25900.0,573585.0,1114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,541909.0,NaN,NaN,NaN,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,NaN,NaN,NaN,2011-07-04 13:34:57.156386,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,NaN,NaN,NaN,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,NaN,NaN,NaN,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Premier diagnostic de qualité
********************************

In [9]:
#   Valeurs manquantes
df.isna().sum().sort_values(ascending=False)

CustomerID     135080
Description      1454
StockCode           0
InvoiceNo           0
Quantity            0
InvoiceDate         0
UnitPrice           0
Country             0
dtype: int64

In [10]:
#   Lignes dupliquées
print("Lignes dupliquées :", df.duplicated().sum())

Lignes dupliquées : 5268


In [11]:
#   Quantités et prix
print("Quantités négatives :", (df["Quantity"] < 0).sum())
print("Prix négatifs :", (df["UnitPrice"] < 0).sum())

Quantités négatives : 10624
Prix négatifs : 2


In [12]:
#   Identifier les commandes annulées
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

cancelled = df[df["InvoiceNo"].str.startswith("C")]

print("Nombre de lignes d'annulation :", len(cancelled))
cancelled.head()

Nombre de lignes d'annulation : 9288


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [13]:
#   Calculer les indicateurs de qualité
quality_report = pd.DataFrame({
    "Valeurs manquantes": df.isna().sum(),
    "Pourcentage manquant (%)": (
        df.isna().mean() * 100
    ).round(2),
    "Valeurs distinctes": df.nunique()
})

quality_report

,Valeurs manquantes,Pourcentage manquant (%),Valeurs distinctes
InvoiceNo,0,0.00,25900
StockCode,0,0.00,4070
Description,1454,0.27,4223
Quantity,0,0.00,722
InvoiceDate,0,0.00,23260
UnitPrice,0,0.00,1630
CustomerID,135080,24.93,4372
Country,0,0.00,38


In [14]:
#    Exporter les données en CSV
output_path = BASE_DIR / "data/raw/online_retail.csv"

df.to_csv(output_path, index=False)

print("Fichier exporté :", output_path)
print("Fichier présent :", output_path.exists())

Fichier exporté : c:\Users\user\Downloads\Retail_BI_Arborescence_Prof (1)\tp2\data\raw\online_retail.csv
Fichier présent : True


Partie 2 — Nettoyage et transformation
*********************************************

In [15]:
#Étape 1 — Supprimer les doublons exacts
df_clean = df.copy()

print("Lignes avant :", len(df_clean))
print("Doublons exacts :", df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates()

print("Lignes après :", len(df_clean))

Lignes avant : 541909
Doublons exacts : 5268
Lignes après : 536641


In [16]:
#Étape 2 — Séparer les annulations
cancelled = df_clean[
    df_clean["InvoiceNo"].str.startswith("C", na=False)
].copy()

ventes = df_clean[
    ~df_clean["InvoiceNo"].str.startswith("C", na=False)
].copy()

print("Annulations :", len(cancelled))
print("Lignes restantes :", len(ventes))

Annulations : 9251
Lignes restantes : 527390


In [17]:
#Étape 3 — Écarter les quantités et prix invalides
print("Quantités négatives :", (ventes["Quantity"] < 0).sum())
print("Prix nuls ou négatifs :", (ventes["UnitPrice"] <= 0).sum())

ventes = ventes[
    (ventes["Quantity"] > 0) &
    (ventes["UnitPrice"] > 0)
].copy()

print("Lignes après filtrage :", len(ventes))

Quantités négatives : 1336
Prix nuls ou négatifs : 2512
Lignes après filtrage : 524878


In [18]:
#Étape 4 — Écarter les codes produit techniques

# Convertir les codes produit en texte
ventes["StockCode"] = ventes["StockCode"].astype("string").str.strip()

# Conserver les codes composés exactement de 5 chiffres
masque_produits = ventes["StockCode"].str.fullmatch(
    r"\d{5}",
    na=False
)

ventes = ventes[masque_produits].copy()

print("Lignes après filtrage :", len(ventes))

Lignes après filtrage : 472253


In [19]:
#Étape 5 — Traiter les clients et les descriptions manquants
print("Clients manquants :", ventes["CustomerID"].isna().sum())

ventes["CustomerID"] = ventes["CustomerID"].fillna(0)

print("Clients manquants après traitement :",
      ventes["CustomerID"].isna().sum())

Clients manquants : 113976
Clients manquants après traitement : 0


In [20]:
#Étape 6 — Normaliser les descriptions et calculer le montant
ventes["Description"] = (
    ventes["Description"].fillna("DESCRIPTION INCONNUE")
    .str.strip()
    .str.upper()
)

ventes["montant"] = ventes["Quantity"] * ventes["UnitPrice"]

ventes[["InvoiceNo", "StockCode", "Description",
        "Quantity", "UnitPrice", "montant"]].head()

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,montant
1,536365,71053,WHITE METAL LANTERN,6,3.39,20.34
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,7.65,15.30
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,4.25,25.50
7,536366,22633,HAND WARMER UNION JACK,6,1.85,11.10
8,536366,22632,HAND WARMER RED POLKA DOT,6,1.85,11.10


In [21]:
#Étape 7 — Enregistrer les données nettoyées
from pathlib import Path

output_dir = BASE_DIR / "data/clean"
output_dir.mkdir(parents=True, exist_ok=True)

ventes.to_csv(output_dir / "ventes.csv", index=False)
cancelled.to_csv(output_dir / "annulations.csv", index=False)

print("Ventes exportées :", len(ventes))
print("Annulations exportées :", len(cancelled))

Ventes exportées : 472253
Annulations exportées : 9251


In [22]:
#Vérifier les données nettoyées
print("Nombre de ventes :", len(ventes))
print("Nombre d'annulations :", len(cancelled))
print("Doublons restants :", ventes.duplicated().sum())
print("Quantités non positives :", (ventes["Quantity"] <= 0).sum())
print("Prix non positifs :", (ventes["UnitPrice"] <= 0).sum())
print("Clients manquants :", ventes["CustomerID"].isna().sum())
print("Descriptions manquantes :", ventes["Description"].isna().sum())

Nombre de ventes : 472253
Nombre d'annulations : 9251
Doublons restants : 0
Quantités non positives : 0
Prix non positifs : 0
Clients manquants : 0
Descriptions manquantes : 0


TP 1.3 : calculer les indicateurs clés (KPI) avec Pandas
**********************************************************

In [23]:
#Étape 1 — Charger les données nettoyées
from pathlib import Path
import pandas as pd

file_path = BASE_DIR / "data/clean/ventes.csv"

print("Dossier courant :", Path.cwd())
print("Fichier trouvé :", file_path.exists())

Dossier courant : c:\Users\user\Downloads\Retail_BI_Arborescence_Prof (1)\tp2\notebooks
Fichier trouvé : True


In [24]:
ventes = pd.read_csv(
    file_path,
    dtype={
        "InvoiceNo": str,
        "StockCode": str
    }
)

print("Dimensions :", ventes.shape)
ventes.head()

Dimensions : (472253, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,montant
0,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
1,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,15.30
2,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,25.50
3,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.10
4,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.10


In [25]:
#Étape 2 — Calculer les premiers KPI

if "montant" not in ventes.columns:
    ventes["montant"] = ventes["Quantity"] * ventes["UnitPrice"]

print("Chiffre d'affaires total :", round(ventes["montant"].sum(), 2), "£")
print("Nombre de lignes :", len(ventes))
print("Nombre de factures :", ventes["InvoiceNo"].nunique())
print("Nombre de produits différents :", ventes["StockCode"].nunique())
print("Nombre de pays :", ventes["Country"].nunique())

Chiffre d'affaires total : 9345497.67 £
Nombre de lignes : 472253
Nombre de factures : 19549
Nombre de produits différents : 2868
Nombre de pays : 38


In [26]:
#Étape 3 — Calculer le chiffre d’affaires par mois
ventes["InvoiceDate"] = pd.to_datetime(ventes["InvoiceDate"])

ventes["mois"] = ventes["InvoiceDate"].dt.to_period("M")

ca_mensuel = (
    ventes.groupby("mois")["montant"]
    .sum()
    .reset_index()
)

ca_mensuel["mois"] = ca_mensuel["mois"].astype(str)

ca_mensuel

,mois,montant
0,2010-12,681867.68
1,2011-01,582246.11
2,2011-02,452685.03
3,2011-03,619860.72
4,2011-04,457415.58
5,2011-05,669162.07
6,2011-06,678033.73
7,2011-07,616710.93
8,2011-08,659840.75
9,2011-09,957036.68
